In [1]:
import os, sys
import numpy as np
import pickle as pk
from random import randint
from itertools import product, chain
import scipy.interpolate as itp
from multiprocessing import Pool, Process

sys.path.append('/home/nishant/lab/MFB/scripts')
sys.path.append('/home/nishant/lab/MFB/steps')
# sys.path.append('/home/nishant/lab/scripts')
from analysis import *
from peaks import *
from misc import *

In [2]:
resultPath = "/media/nishant/data/results/findPr"
dirs = np.array(getDirs(resultPath, sstr='nVDCC'))

# dirs = []
# nvdcc = range(2, 15)
# dvdcc = range(60, 230, 20)
# naz   = [31, 33, 35] #range(7, 30, 2)
# for nVDCC, dVDCC, nAZ in product(nvdcc, dvdcc, naz):
#     dirs.append(f"nVDCC_{nVDCC}_dVDCC_{dVDCC}_nAZ_{nAZ}")

# dirs = []
# finfo = np.genfromtxt('/home/nishant/lab/MFB/mcell/findPr/newsims', dtype=int)
# for fi in finfo:
#     dirs.append(f"nVDCC_{fi[0]}_dVDCC_{fi[1]}_nAZ_{fi[2]}")

print(len(dirs))
tempdirs = [a for a in dirs if 'nVDCC_' in a]
for i,d in enumerate(tempdirs):
    print(i, d)


2025
0 nVDCC_10_dVDCC_100_nAZ_11
1 nVDCC_10_dVDCC_100_nAZ_13
2 nVDCC_10_dVDCC_100_nAZ_15
3 nVDCC_10_dVDCC_100_nAZ_17
4 nVDCC_10_dVDCC_100_nAZ_19
5 nVDCC_10_dVDCC_100_nAZ_21
6 nVDCC_10_dVDCC_100_nAZ_23
7 nVDCC_10_dVDCC_100_nAZ_25
8 nVDCC_10_dVDCC_100_nAZ_27
9 nVDCC_10_dVDCC_100_nAZ_29
10 nVDCC_10_dVDCC_100_nAZ_31
11 nVDCC_10_dVDCC_100_nAZ_33
12 nVDCC_10_dVDCC_100_nAZ_35
13 nVDCC_10_dVDCC_100_nAZ_7
14 nVDCC_10_dVDCC_100_nAZ_9
15 nVDCC_10_dVDCC_120_nAZ_11
16 nVDCC_10_dVDCC_120_nAZ_13
17 nVDCC_10_dVDCC_120_nAZ_15
18 nVDCC_10_dVDCC_120_nAZ_17
19 nVDCC_10_dVDCC_120_nAZ_19
20 nVDCC_10_dVDCC_120_nAZ_21
21 nVDCC_10_dVDCC_120_nAZ_23
22 nVDCC_10_dVDCC_120_nAZ_25
23 nVDCC_10_dVDCC_120_nAZ_27
24 nVDCC_10_dVDCC_120_nAZ_29
25 nVDCC_10_dVDCC_120_nAZ_31
26 nVDCC_10_dVDCC_120_nAZ_33
27 nVDCC_10_dVDCC_120_nAZ_35
28 nVDCC_10_dVDCC_120_nAZ_7
29 nVDCC_10_dVDCC_120_nAZ_9
30 nVDCC_10_dVDCC_140_nAZ_11
31 nVDCC_10_dVDCC_140_nAZ_13
32 nVDCC_10_dVDCC_140_nAZ_15
33 nVDCC_10_dVDCC_140_nAZ_17
34 nVDCC_10_dVDCC_140_n

## AZ simulation in STEPS

In [3]:
from MFB_model import *
mdl, sim, r = get_MFB_model()

In [4]:
def runAZTrials(CaData, RRPs):
    vesData = []
    resAZs = []
    # print(CaData[0], CaData[1], RRPs)
    for RRP in RRPs:
        # resCa, resAZ, vesRel = simAZ(CaData[0], CaData[1], sim, r, RRP=RRP)
        resAZ, vesRel = simAZ(CaData[0], CaData[1], sim, r, RRP=RRP)
        vesRelTot = np.sum(vesRel, axis=1)
        pks = detect_peaks(vesRelTot, edge='rising', show=False)
        vesData.append(list(CaData[0,pks]))
        # resAZs.append(resAZ)
    
    return [vesData, resAZs]

In [5]:
def getVesRel(dir, RRPs, resultPath, fname='CaConc.dat', trials=2000):
    nAZ = int(getSimInfo(dir, 'nAZ'))
    rrps = len(RRPs)
    
    CaFile = os.path.join(resultPath, dir, fname)
    CaData = np.genfromtxt(CaFile, unpack=True) # in uM
    
    p = Pool(5)
    vesRelTimes = []
    vesRels = []
    for iCa in tq(range(1,nAZ+1), desc=dir):
    # for iCa in range(1,nAZ+1):
        
        info = product([CaData[(0,iCa),:]], [RRPs]*trials)
        # print(list(info)[0])
        vRelTime = np.array(p.starmap(runAZTrials, info), dtype=object)
        # vRelTime = runAZTrials(CaData[(0,iCa),:], RRPs)
        # print(vRelTime[:,0])

        vesRelTime = vRelTime[:,0]
        vesRelTime = [[vesRelTime[i][j] for i in range(trials)] for j in range(rrps)]
        vesRelTimes.append(vesRelTime)
        #print(vesRelTime[:,1])

        # AZstates = vRelTime[:,1]
        # AZstates = np.mean(AZstates, axis=0)
        #print(AZstates[0])
        #print(np.vstack((CaData[0,],AZstates[0])))
        # for i,rrp in enumerate(RRPs):
        #     fAZ = os.path.join(resultPath, dir, f'AZ_{iCa}_RRP_{rrp}.dat')
        #     AZdata = np.hstack((np.array([CaData[0]]).T, AZstates[i]))
        #     np.savetxt(fAZ, AZdata, fmt=['%0.5f']+['%0.3f']*18, delimiter="\t")

    p.close()
    p.join()
    
    vesRelTimes = [[list(chain(*[vesRelTimes[i][j][k] for i in range(nAZ)])) for k in range(trials)] for j in range(rrps)]
    vesData = {}
    for RRP,v in zip(RRPs, vesRelTimes):
        vesData.update({str(RRP): v})
    
    return vesData

# dir = tempdirs[0] 
# rrp = 30 #int(getSimInfo(dir, 'RRP'))
# vesData = getVesRel(dir, RRPs=[rrp], resultPath=resultPath, trials=1000)

In [6]:
with open(os.path.join(resultPath, 'correctionSim'), "rb") as infile:
    correctionSim = pk.load(infile)

correctionSim

[['nVDCC_10_dVDCC_100_nAZ_31', [60]],
 ['nVDCC_10_dVDCC_100_nAZ_33', [60]],
 ['nVDCC_10_dVDCC_100_nAZ_35', [60]],
 ['nVDCC_10_dVDCC_120_nAZ_31', [60]],
 ['nVDCC_10_dVDCC_120_nAZ_33', [60]],
 ['nVDCC_10_dVDCC_120_nAZ_35', [60]],
 ['nVDCC_10_dVDCC_140_nAZ_31', [60]],
 ['nVDCC_10_dVDCC_140_nAZ_33', [60]],
 ['nVDCC_10_dVDCC_140_nAZ_35', [60]],
 ['nVDCC_10_dVDCC_160_nAZ_31', [60]],
 ['nVDCC_10_dVDCC_160_nAZ_33', [60]],
 ['nVDCC_10_dVDCC_160_nAZ_35', [60]],
 ['nVDCC_10_dVDCC_180_nAZ_31', [60]],
 ['nVDCC_10_dVDCC_180_nAZ_33', [60]],
 ['nVDCC_10_dVDCC_180_nAZ_35', [60]],
 ['nVDCC_10_dVDCC_200_nAZ_31', [60]],
 ['nVDCC_10_dVDCC_200_nAZ_33', [60]],
 ['nVDCC_10_dVDCC_200_nAZ_35', [60]],
 ['nVDCC_10_dVDCC_60_nAZ_17', [45, 50, 55, 60]],
 ['nVDCC_10_dVDCC_60_nAZ_19', [45, 50, 55, 60]],
 ['nVDCC_10_dVDCC_60_nAZ_21', [45, 50, 55, 60]],
 ['nVDCC_10_dVDCC_60_nAZ_23', [45, 50, 55, 60]],
 ['nVDCC_10_dVDCC_60_nAZ_25', [45, 50, 55, 60]],
 ['nVDCC_10_dVDCC_60_nAZ_27', [45, 50, 55, 60]],
 ['nVDCC_10_dVDCC_60_n

In [ ]:
for dir, RRP in correctionSim:
    try:
        with open(os.path.join(resultPath, dir, 'vesData.dat'), 'rb') as file:
            vesData = pk.load(file)
            newVesData = getVesRel(dir, RRPs=RRP, resultPath=resultPath, trials=1000)
            vesData.update(newVesData)
        with open(os.path.join(resultPath, dir, 'vesData.dat'), "wb") as outfile:
            pk.dump(vesData, outfile)

    except FileNotFoundError:
        try:
            vesData = getVesRel(dir, RRPs=range(5,61,5), resultPath=resultPath, trials=1000)
            with open(os.path.join(resultPath, dir, 'vesData.dat'), "wb") as outfile:
                pk.dump(vesData, outfile)
        except:
            print(f'Error in {dir}')
    
    except:
        print(f'Error in {dir}')


## Get Pr Statistics

In [ ]:
info = [[d, resultPath] for d in tempdirs]

p = Pool(38)
p.starmap(PrStat, info)

# for d in tq(tempdirs[:30]):
#     PrStat(d, resultPath)

## Get Vesicle Release Statistics

In [ ]:
for d in tq(tempdirs[:]):
    try:
        relStat(d, resultPath, resample=1000)
    except:
        print(f'Error in {d}')


## Plotting Data

In [ ]:
nFig = 3
figure, ax = plt.subplots(nFig, figsize=(10, 3*nFig), sharex=True)
#figure.tight_layout()
figure.subplots_adjust(hspace=0.0, top=.95)

for d in tempdirs[1:].tolist()+tempdirs[:1].tolist():
    fname = os.path.join(resultPath, d, 'result_RRP_35.dat')
    relData = np.genfromtxt(fname, unpack=True)

    ax[0].plot(range(1,len(relData[0])+1), relData[0], 'o-', label=getSimInfo(d, 'ISI'))
    ax[0].set_ylabel('Pr')
    ax[0].legend(frameon=False)
    
    ax[1].plot(range(1,len(relData[0])+1), relData[0]/relData[0][0], 'o-', label=getSimInfo(d, 'ISI'))
    ax[1].set_ylabel('$Pr_n/Pr_1$')
    ax[1].legend(frameon=False)
    
    ax[2].plot(range(1,len(relData[0])+1), relData[2], 'o-', label=getSimInfo(d, 'ISI'))
    ax[2].set_ylabel('Releases per trial')
    ax[2].set_xlabel('Stimulus number')
    ax[2].legend(frameon=False)

figure.suptitle("nVDCC_5_dVDCC_100_nAZ_18")
figure.savefig('nVDCC_5_dVDCC_100_nAZ_18.eps', fmt='eps', dpi=300, transparent=True)

## Testing a single trial with STEPS

In [ ]:
CaFile = resultPath + 'nVDCC_9_dVDCC_60_nAZ_29/CaConc.dat'
CaData = np.genfromtxt(CaFile, unpack=True)

T = np.arange(CaData[0][0], CaData[0][-1], 1e-5)
CaInterp = itp.interp1d(CaData[0], CaData[1])
CaInterpData = CaInterp(T)

resAZ, vesRel = simAZtest(T, CaInterpData, sim, r)

nFig = 4
figure, ax = plt.subplots(nFig, figsize=(15, 4*nFig), sharex=True)
figure.subplots_adjust(hspace=0.0)
labelfontsize = 13

T=T*1e3
ax[0].plot(T, CaInterpData, label='Ca')
ax[0].set_ylabel(r'Ca ($\mu M$)', fontsize=labelfontsize)

for mol in azMolName[:18]:
    i = azMolName.index(mol)
    ax[1].plot(T, resAZ[:,i], label=mol)
ax[1].set_ylabel('AZ states (num)', fontsize=labelfontsize)
ax[1].legend(bbox_to_anchor=(1, 1), loc='upper left', ncol=2)

vesRelSync  = np.sum(vesRel[:,:3], axis=1)
vesRelAsync = np.sum(vesRel[:,3:-1], axis=1)
vesRelSpont = vesRel[:,-1]
vesRelTot   = np.sum(vesRel, axis=1)

ax[2].plot(T, vesRelSync, label='sync')
ax[2].plot(T, vesRelAsync, label='async')
ax[2].plot(T, vesRelSpont, label='spont')
ax[2].plot(T, vesRelTot, label='total')
    
ax[2].set_ylabel('Vesicles released', fontsize=labelfontsize)

ax[2].legend()

peaks = detect_peaks(vesRelTot, edge='rising', show=False)
ax[2].plot(T[peaks], vesRelTot[peaks], '+', mfc=None, mec='r', mew=2, ms=15)
#print(T[peaks])

rrp = np.sum(resAZ, axis=1)
ax[3].plot(T, rrp, label='total')
ax[3].set_ylabel('Vesicles released', fontsize=labelfontsize)
ax[3].set_xlabel('Time (ms)', fontsize=labelfontsize)

plt.show()